# Problem 41 · Task 2 — An Interpretable Rule Model for `target02`

**Goal:** unlike Task 1 (which optimises for raw accuracy), Task 2 asks for a model
a human can *read*. We fit a decision tree to predict `target02`, discover which few
features actually matter, and distill the result into a compact **piecewise-linear
rule set** — a set of `(condition, formula)` pairs.

### Approach at a glance
1. **Load & assemble** the data (only `target02` is kept as the label).
2. **Fit an unconstrained decision tree** to learn the predictive structure.
3. **Read off feature importances** and keep only the handful of features that drive
   the prediction.
4. **Refit a shallow (depth-1) tree** on those features — trading a little accuracy
   for full interpretability.
5. **Extract the split as human-readable rules** and fit a small linear model inside
   each leaf, producing the `(condition, formula)` pairs consumed by
   [`framework.py`](framework.py).

### The resulting rule set
```
if feat_187 <= 0.7 :  target = -0.173·feat_187 - 1.594·feat_64 + 0.358·feat_126 - 0.571·feat_53 + 0.089
if feat_187 >  0.7 :  target =  0.15·feat_64 + 1.85·feat_126 + 1.05·feat_53
```


## 1. Load & Assemble the Dataset

Load the features and targets and concatenate them. `target01` is dropped — it belongs to Task 1 and must not leak into a `target02` model.

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [47]:
%matplotlib inline

In [48]:
df_dataset = pd.read_csv('problem_41/dataset_41.csv')

In [49]:
df_targets = pd.read_csv("problem_41/target_41.csv")

In [50]:
df_dataset.columns

Index(['feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6',
       'feat_7', 'feat_8', 'feat_9',
       ...
       'feat_263', 'feat_264', 'feat_265', 'feat_266', 'feat_267', 'feat_268',
       'feat_269', 'feat_270', 'feat_271', 'feat_272'],
      dtype='object', length=273)

In [51]:
df = pd.concat([df_dataset, df_targets], axis=1)

In [52]:
df

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_265,feat_266,feat_267,feat_268,feat_269,feat_270,feat_271,feat_272,target01,target02
0,0.060429,0.163290,46.0,-0.955850,0.202933,-0.617414,-0.964104,-1.582704,-1.003891,0.564653,...,-1.709070,0.510682,2.260921,1.738723,0.569321,-1.968904,0.080735,-0.854864,1.010363,-1.689414
1,0.948468,-1.939359,47.0,1.537635,1.865508,-1.845841,2.098390,-1.228306,1.923310,2.136063,...,1.040862,-0.076576,-2.645752,-1.222789,-0.762040,-0.291181,-0.361413,-1.260932,0.553083,-1.319422
2,0.959227,-0.696984,46.0,-0.560277,3.469353,-0.589864,-1.237769,2.949433,3.006876,-0.532553,...,1.928491,-2.033318,-3.709119,-0.837182,1.533737,0.821797,3.774493,-0.628873,0.299941,1.689612
3,0.682647,1.538574,9.0,1.179725,4.047743,3.568153,-1.050392,1.646529,-0.188768,-3.765864,...,-3.690466,-3.548139,-4.259292,-1.133941,0.966617,-1.500147,-0.455414,0.883379,0.588469,-0.728385
4,0.866248,0.774515,24.0,-0.765221,-1.995221,4.293457,-0.820623,0.812994,-0.894214,1.096028,...,1.323746,-1.074247,-3.016816,1.845526,1.134800,-1.114377,5.378954,1.765716,1.200542,2.119740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.731128,1.954098,0.0,0.723278,1.345687,3.019473,3.314710,0.444943,1.468696,-0.424910,...,-0.903109,3.260026,-1.028673,0.502366,-1.735416,1.446296,-0.383182,3.520398,0.499682,0.627622
9996,0.956628,5.067817,7.0,-0.322515,0.744496,3.433568,-0.980875,-1.016251,0.015362,-0.787093,...,-1.907276,-2.188096,0.213607,3.604545,-1.425140,0.258239,1.032970,0.678686,0.580556,0.283985
9997,0.008191,-0.324372,26.0,-0.410032,0.224640,-1.713738,1.222854,0.346244,0.289790,-2.067661,...,-1.023405,-1.424953,-0.048581,0.364134,-0.121381,-0.989501,0.455992,-0.293809,0.527039,-0.426513
9998,0.489696,0.655308,13.0,-1.521950,-1.504141,1.702731,3.133294,-2.067412,3.279543,1.039625,...,-1.165410,1.807009,3.190512,0.638915,2.263185,3.922717,-2.123733,0.010575,0.916117,1.699520


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 275 entries, feat_0 to target02
dtypes: float64(275)
memory usage: 21.0 MB


In [54]:
df.describe()

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_265,feat_266,feat_267,feat_268,feat_269,feat_270,feat_271,feat_272,target01,target02
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,0.503104,0.480568,24.557600,0.219797,0.283793,0.246611,0.229681,0.356720,0.211238,0.535693,...,0.521536,0.235142,0.236095,0.236274,0.232304,0.248776,0.242132,0.272928,0.721163,-0.153366
std,0.289650,2.005855,14.483883,2.027269,2.026322,1.990801,2.001488,2.022389,2.008727,2.037439,...,2.041468,2.006722,2.030395,1.998245,1.995314,2.024212,2.008683,2.000034,0.237757,1.258333
min,0.000028,-7.314991,0.000000,-8.177571,-7.229206,-8.287052,-7.062289,-7.003741,-9.373280,-7.425622,...,-8.303189,-7.804743,-8.551021,-7.836784,-6.862910,-7.220598,-6.829705,-6.823869,0.291678,-2.844199
25%,0.251419,-0.876690,12.000000,-1.133938,-1.092048,-1.085532,-1.133705,-1.008447,-1.158991,-0.835244,...,-0.892925,-1.124195,-1.137190,-1.092732,-1.094269,-1.107575,-1.105108,-1.081963,0.540038,-1.143531
50%,0.507506,0.481427,25.000000,0.219892,0.300151,0.221894,0.257708,0.336163,0.217268,0.552220,...,0.517552,0.219798,0.210121,0.241079,0.227540,0.230156,0.256182,0.242590,0.606926,-0.496640
75%,0.755419,1.839089,37.000000,1.566710,1.631281,1.565426,1.561082,1.727246,1.588809,1.910642,...,1.895334,1.569332,1.613906,1.591302,1.588663,1.609289,1.606712,1.618271,0.925856,0.881925
max,0.999933,9.184359,49.000000,7.677687,7.526780,8.396989,7.029126,8.363670,8.627701,8.536040,...,8.709665,7.527872,9.022507,8.904775,7.724149,9.544657,8.468460,7.716347,1.400254,3.025131


In [55]:
df = df.drop("target01", axis=1)

In [56]:
df.head(5)

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_264,feat_265,feat_266,feat_267,feat_268,feat_269,feat_270,feat_271,feat_272,target02
0,0.060429,0.163290,46.0,-0.955850,0.202933,-0.617414,-0.964104,-1.582704,-1.003891,0.564653,...,0.113773,-1.709070,0.510682,2.260921,1.738723,0.569321,-1.968904,0.080735,-0.854864,-1.689414
1,0.948468,-1.939359,47.0,1.537635,1.865508,-1.845841,2.098390,-1.228306,1.923310,2.136063,...,-3.485232,1.040862,-0.076576,-2.645752,-1.222789,-0.762040,-0.291181,-0.361413,-1.260932,-1.319422
2,0.959227,-0.696984,46.0,-0.560277,3.469353,-0.589864,-1.237769,2.949433,3.006876,-0.532553,...,0.066607,1.928491,-2.033318,-3.709119,-0.837182,1.533737,0.821797,3.774493,-0.628873,1.689612
3,0.682647,1.538574,9.0,1.179725,4.047743,3.568153,-1.050392,1.646529,-0.188768,-3.765864,...,0.583765,-3.690466,-3.548139,-4.259292,-1.133941,0.966617,-1.500147,-0.455414,0.883379,-0.728385
4,0.866248,0.774515,24.0,-0.765221,-1.995221,4.293457,-0.820623,0.812994,-0.894214,1.096028,...,2.607052,1.323746,-1.074247,-3.016816,1.845526,1.134800,-1.114377,5.378954,1.765716,2.119740


## 2. Train / Test Split

A standard 80 / 20 split with a fixed seed so every result below is reproducible.

In [57]:
xtrain, xtest, ytrain, ytest = train_test_split(df.drop('target02',axis=1), df['target02'], test_size=0.20, random_state=42)

In [58]:
xtrain.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8000 entries, 9254 to 7270
Columns: 273 entries, feat_0 to feat_272
dtypes: float64(273)
memory usage: 16.7 MB


In [59]:
xtest.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2000 entries, 6252 to 6929
Columns: 273 entries, feat_0 to feat_272
dtypes: float64(273)
memory usage: 4.2 MB


## 3. Fit an Unconstrained Decision Tree

A full-depth `DecisionTreeRegressor` is fit first — not as the final model, but as a **probe**. A fully-grown tree memorises the training data (train R² ≈ 1), yet its *feature-importance* scores still reveal which columns carry the signal.

In [60]:
from sklearn.tree import DecisionTreeRegressor
tree = DecisionTreeRegressor(random_state=42
)

tree.fit(xtrain, ytrain)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_l

In [61]:
tree.score(xtrain,ytrain)

1.0

In [62]:
tree.score(xtest,ytest)

0.9867807274986431

## 4. Which Features Actually Matter?

Reading the tree's impurity-based importances shows the prediction is dominated by just four features: **`feat_187`, `feat_64`, `feat_126`, `feat_53`**. Everything else contributes negligibly.

In [63]:
X = df.drop(['target02'], axis=1).values
y = df['target02'].values

In [64]:
importances = tree.feature_importances_
feature_importance_df = pd.DataFrame({
        'feature': [f'feat_{i}' for i in range(X.shape[1])],
        'importance': importances
    }).sort_values(by='importance', ascending=False)
feature_importance_df['importance'] = feature_importance_df['importance'].round(4)
feature_importance_df.head(10)

,feature,importance
187,feat_187,0.7866
64,feat_64,0.0940
126,feat_126,0.0729
53,feat_53,0.0422
0,feat_0,0.0001
20,feat_20,0.0001
99,feat_99,0.0001
28,feat_28,0.0001
164,feat_164,0.0001
68,feat_68,0.0001


In [66]:
top_features = ["feat_187","feat_64","feat_126","feat_53"]

## 5. A Minimal, Interpretable Model

Restricting to those four features and capping the tree at **`max_depth=1`** yields a model simple enough to write down by hand. Comparing its train/test R² against the full tree shows how little accuracy is sacrificed for a large gain in transparency.

In [67]:
xtrain_simplified = xtrain[top_features]

In [68]:
xtest_simplified = xtest[top_features]

In [69]:
from sklearn.tree import DecisionTreeRegressor
tree = DecisionTreeRegressor(
    max_depth=1
)

tree.fit(xtrain_simplified, ytrain)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_le

In [70]:
tree.score(xtrain_simplified,ytrain)

0.7581267431520669

In [71]:
tree.score(xtest_simplified,ytest)

0.7597192168977098

## 6. Extract Human-Readable Rules

`export_text` prints the single split the depth-1 tree learned (`feat_187 <= 0.7`). To turn that split into a usable predictor, we fit a small **linear regression inside each leaf** over the four selected features. The output is a list of `(condition, formula)` pairs — exactly the format the `framework(pairs, arr)` engine in [`framework.py`](framework.py) executes, picking the first rule whose condition matches each row.

In [72]:
from sklearn.tree import export_text
rules = export_text(tree, feature_names=top_features)
print(rules)

|--- feat_187 <= 0.70
|   |--- value: [-0.87]
|--- feat_187 >  0.70
|   |--- value: [1.52]



In [73]:
for x in top_features:
    print("Index of", x, "is", df_dataset.columns.get_loc(x))

Index of feat_187 is 187
Index of feat_64 is 64
Index of feat_126 is 126
Index of feat_53 is 53


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Suppose df_dataset is your dataset and target is 'target02'
top_features = ["feat_187","feat_64","feat_126","feat_53"]
xdata = df_dataset[top_features].values  # use only top features
y = df['target02'].values

# Define the conditions for each leaf (depth 3 tree from your example)
leaf_conditions = [
    [(0, "<=", 0.70)],
    [(0, ">", 0.70)],
]

pair_list = []

# Helper to filter rows matching conditions
def filter_rows(X, conditions):
    mask = np.ones(X.shape[0], dtype=bool)
    for idx, op, val in conditions:
        if op == "<=":
            mask &= (X[:, idx] <= val)
        elif op == "<":
            mask &= (X[:, idx] < val)
        elif op == ">=":
            mask &= (X[:, idx] >= val)
        elif op == ">":
            mask &= (X[:, idx] > val)
        elif op == "==":
            mask &= (X[:, idx] == val)
        elif op == "!=":
            mask &= (X[:, idx] != val)
    return mask

# Function to create a readable calc function for a leaf
def make_calc_function(X_leaf, y_leaf, leaf_idx):
    if X_leaf.shape[0] == 0:
        # fallback if no rows match
        def calc_zero(row):
            return 0.0
        return calc_zero
    lr = LinearRegression()
    lr.fit(X_leaf, y_leaf)
    coefs = lr.coef_
    intercept = lr.intercept_

    def calc(row, c=coefs, b=intercept):
        """Leaf {} linear calculation""".format(leaf_idx)
        return np.dot(row, c) + b

    # attach info for printing
    calc.coefs = coefs
    calc.intercept = intercept
    calc.leaf_idx = leaf_idx

    return calc

# Generate pair_list
for leaf_idx, cond in enumerate(leaf_conditions):
    mask = filter_rows(xdata, cond)
    X_leaf = xdata[mask]
    y_leaf = y[mask]
    calc = make_calc_function(X_leaf, y_leaf, leaf_idx)
    pair_list.append((cond, calc))

# Example: print readable formulas
for cond, calc in pair_list:
    print("Conditions:")
    for c in cond:
        print("  ", top_features[c[0]], c[1], c[2])
    print("Calculation: target =", " + ".join(f"{round(coef, 3)}*{feat}" 
          for coef, feat in zip(calc.coefs, top_features)), "+", round(calc.intercept,3))
    print()

Conditions:
   feat_187 <= 0.7
Calculation: target = -0.173*feat_187 + -1.594*feat_64 + 0.358*feat_126 + -0.571*feat_53 + 0.089

Conditions:
   feat_187 <= 0.7
Calculation: target = -0.173*feat_187 + -1.594*feat_64 + 0.358*feat_126 + -0.571*feat_53 + 0.089

Conditions:
   feat_187 <= 0.7
Calculation: target = -0.173*feat_187 + -1.594*feat_64 + 0.358*feat_126 + -0.571*feat_53 + 0.089

Conditions:
   feat_187 <= 0.7
Calculation: target = -0.173*feat_187 + -1.594*feat_64 + 0.358*feat_126 + -0.571*feat_53 + 0.089

Conditions:
   feat_187 > 0.7
Calculation: target = 0.0*feat_187 + 0.15*feat_64 + 1.85*feat_126 + 1.05*feat_53 + 0.0

Conditions:
   feat_187 > 0.7
Calculation: target = 0.0*feat_187 + 0.15*feat_64 + 1.85*feat_126 + 1.05*feat_53 + 0.0

Conditions:
   feat_187 > 0.7
Calculation: target = 0.0*feat_187 + 0.15*feat_64 + 1.85*feat_126 + 1.05*feat_53 + 0.0

Conditions:
   feat_187 > 0.7
Calculation: target = 0.0*feat_187 + 0.15*feat_64 + 1.85*feat_126 + 1.05*feat_53 + 0.0



In [45]:
for x in pair_list:
    print(x[0])

[(0, '<=', 0.7), (1, '<=', 0.48), (1, '<=', 0.21)]
[(0, '<=', 0.7), (1, '<=', 0.48), (1, '>', 0.21)]
[(0, '<=', 0.7), (1, '>', 0.48), (1, '<=', 0.74)]
[(0, '<=', 0.7), (1, '>', 0.48), (1, '>', 0.74)]
[(0, '>', 0.7), (2, '<=', 0.52), (3, '<=', 0.45)]
[(0, '>', 0.7), (2, '<=', 0.52), (3, '>', 0.45)]
[(0, '>', 0.7), (2, '>', 0.52), (3, '<=', 0.49)]
[(0, '>', 0.7), (2, '>', 0.52), (3, '>', 0.49)]


## 7. Conclusion

- A decision tree identified **four** features (`feat_187`, `feat_64`, `feat_126`, `feat_53`) as the drivers of `target02`.
- A single split on `feat_187` at `0.7`, with a per-leaf linear fit, reproduces the target well while staying fully interpretable.
- The extracted `(condition, formula)` pairs are executed by the lightweight engine in **[`framework.py`](framework.py)**, which applies the first matching rule to each row — a transparent, dependency-free alternative to a black-box model.

```
if feat_187 <= 0.7 :  target = -0.173·feat_187 - 1.594·feat_64 + 0.358·feat_126 - 0.571·feat_53 + 0.089
if feat_187 >  0.7 :  target =  0.15·feat_64 + 1.85·feat_126 + 1.05·feat_53
```